In [3]:
import yfinance as yf 
from bcb import sgs # API do BC 
import pandas as pd
from functools import reduce
from datetime import timedelta, datetime
import sys, os 


df_yf = pd.read_csv("../data/raw/yf_data.csv", parse_dates=["Date"])
df_bcb = pd.read_csv("../data/raw/bcb_data.csv", parse_dates=["Date"])
df_fed = pd.read_csv("../data/raw/fed_data.csv", parse_dates=["DATE"])


In [6]:
# 1) YF: variação mensal acumulada (composta) de dólar/petróleo
var_cols = [c for c in df_yf.columns if c.endswith("_var_pct")]
yf = df_yf[["Date"] + var_cols].copy().set_index("Date").sort_index()

# de % para decimal e compõe no mês: (1+r_dia).prod() - 1
r = yf[var_cols] / 100.0
yf_mensal = (1.0 + r).resample("M").apply(lambda x: (1.0 + x).prod() - 1.0)
yf_mensal = (yf_mensal * 100.0).rename(
    columns=lambda c: c.replace("_var_pct", "_var_mensal_acum_pct")
).reset_index()


/tmp/ipykernel_900/3277224535.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  yf_mensal = (1.0 + r).resample("M").apply(lambda x: (1.0 + x).prod() - 1.0)


In [7]:
# 2) BCB: padroniza para mensal (fim do mês)
bcb = df_bcb.drop_duplicates(subset="Date", keep="last").set_index("Date").sort_index()

# Se sua base tiver séries com frequências diferentes, isso traz a última observação disponível do mês
bcb_mensal = bcb.resample("D").ffill().resample("M").last()

# --- 2a) SELIC efetiva (% a.d.) -> retorno mensal composto (% no mês)
if "SELIC" in bcb_mensal.columns:
    selic_d = bcb["SELIC"] / 100.0                        # usa diário (antes do resample mensal)
    selic_m = (1.0 + selic_d).resample("M").apply(lambda x: (1.0 + x).prod() - 1.0) * 100.0
    # garante alinhamento por data
    bcb_mensal["SELIC_mensal_acum_pct"] = selic_m

bcb_mensal = bcb_mensal.reset_index()

/tmp/ipykernel_900/2045394219.py:5: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  bcb_mensal = bcb.resample("D").ffill().resample("M").last()
/tmp/ipykernel_900/2045394219.py:10: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  selic_m = (1.0 + selic_d).resample("M").apply(lambda x: (1.0 + x).prod() - 1.0) * 100.0


In [8]:
# 3) FED (FRED): traz pra mensal (fim do mês) e renomeia
fed = df_fed.rename(columns={"DATE": "Date"}).copy()
fed = fed.sort_values("Date").set_index("Date")

# Se sua série já vier mensal, resample("M").last() mantém igual; se vier diária, agrega p/ fim do mês
fed_mensal = fed.resample("M").last().reset_index()
# Padroniza o nome da coluna principal se necessário (ajuste aqui pro nome real da sua coluna)
# Ex.: se a coluna chama 'FEDFUNDS' no df_fed:
if "FEDFUNDS" in fed_mensal.columns:
    fed_mensal = fed_mensal.rename(columns={"FEDFUNDS": "FED"})

/tmp/ipykernel_900/1933215868.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  fed_mensal = fed.resample("M").last().reset_index()


In [ ]:
# 4) Merge final (mensal)
df_final = (
    bcb_mensal
    .merge(yf_mensal, on="Date", how="outer")
    .merge(fed_mensal, on="Date", how="left")
    .sort_values("Date")
    .reset_index(drop=True)
)

#Transforma a coluna 'Date' em índice datetime e ordena
df_final = df_final.reset_index()
df_final["Date"] = pd.to_datetime(df_final["Date"], errors="coerce")
df_final = df_final.dropna(subset=["Date"]).sort_values("Date")
df_final = df_final.set_index("Date")


# Reordena as colunas para que 'SELIC_META' fique no início, se existir
if "SELIC_META" in df_final.columns:
    cols = ["SELIC_META"] + [c for c in df_final.columns if c != "SELIC_META"]
    df_final = df_final[cols]


df_final

,SELIC_META,index,IPCA,SELIC,IND_DESMP,IPCA_ALIMENTOS,IC-Br Agropecuária,PIB,DIVIDA_EXTERNA,SELIC_mensal_acum_pct,CL=F_var_mensal_acum_pct,USDBRL=X_var_mensal_acum_pct,FED
Date,,,,,,,,,,,,,
2004-01-31,16.50,0,0.76,0.059906,NaN,0.88,99.53,142455.0,154073.00,2.110402e+08,2.595554e+07,0.000000e+00,1.00
2004-02-29,16.50,1,0.61,0.059940,NaN,0.15,102.04,141654.7,152530.50,2.635607e+07,5.490765e+07,0.000000e+00,1.01
2004-03-31,16.25,2,0.47,0.059291,NaN,0.43,106.19,160673.9,150254.75,8.446262e+08,8.350183e+08,0.000000e+00,1.00
2004-04-30,16.00,3,0.37,0.058229,NaN,-0.34,108.50,159113.9,140809.68,1.054754e+08,2.146598e+08,0.000000e+00,1.00
2004-05-31,16.00,4,0.51,0.058160,NaN,0.23,118.01,160112.1,151100.02,2.109988e+08,1.084227e+08,0.000000e+00,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-06-30,13.75,233,-0.08,0.050788,8.0,-0.66,378.28,905466.4,-803200.30,2.108363e+08,2.139094e+08,4.119348e+08,5.08
2023-07-31,13.75,234,0.12,0.050788,7.9,-0.46,384.11,921314.5,-803989.15,2.108363e+08,1.129162e+08,2.068646e+08,5.12
2023-08-31,13.25,235,0.23,0.049037,7.8,-0.85,393.00,932277.4,-836394.16,8.436188e+08,8.487019e+08,8.535405e+08,5.33


In [16]:
df_final.to_csv("../data/processed/dataset_final.csv", index=True)